# Stage 2: Feature Engineering (Beginner's Guide)
### E-Commerce Intelligence & Decision Support Platform

---

### What is "Feature Engineering"?
In **Stage 1 (`DATA_preprocessing.ipynb`)**, we cleaned over 1,000,000 transaction rows.
However, a machine learning model cannot easily understand raw transaction lines like:
> *"Customer 13085 bought 12 units of item 85048 on Tuesday at 07:45 AM for £6.95."*

Instead, we need to calculate **smart summaries** (called **features**):
1. **Customer Features:** How active is this customer? How much do they spend? How often do they buy?
2. **Product Features:** Which products are bestsellers? Which generate the most revenue?
3. **Interaction Features:** Has Customer A bought Product B before? How many times?

Let's build these step-by-step using simple Pandas!


### Step 1: Import Libraries and Load Cleaned Data
We load the clean transactions file created in Stage 1 (`cleaned_transactions.csv`).

In [1]:
import pandas as pd
import numpy as np
import os

# Locate data folder whether running from project root or inside notebooks/
data_path = "data/processed/cleaned_transactions.csv"
if not os.path.exists(data_path):
    data_path = "../data/processed/cleaned_transactions.csv"

# Load transactions
df = pd.read_csv(
    data_path,
    dtype={"InvoiceNo": str, "StockCode": str},
    parse_dates=["InvoiceDate"],
    low_memory=False
)

# We only keep rows with a valid CustomerID for customer modeling
df = df[df["CustomerID"].notnull()].copy()
df["CustomerID"] = df["CustomerID"].astype(int)

print(f"Total valid transactions loaded: {len(df):,}")
print(f"Unique customers: {df['CustomerID'].nunique():,}")
print(f"Unique products:  {df['StockCode'].nunique():,}")
df.head(3)


Total valid transactions loaded: 779,425
Unique customers: 5,878
Unique products:  4,631


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalAmount
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0


### Step 2: The Time Split (Dec 2009–May 2011 vs Jun–Dec 2011)

#### Why do we split by time? (Preventing "Data Leakage")
Imagine studying for a final exam. If someone gives you the exact questions and answers the day before, you might score 100%, but you didn't actually learn!
In Machine Learning:
- **Historical Period (Dec 2009 – May 2011):** The "past". We use ONLY this period to create our features and train models.
- **Evaluation Period (Jun 2011 – Dec 2011):** The "future". Kept locked away to test if our recommendations actually work!


In [2]:
split_date = pd.Timestamp("2011-06-01")

# Split transactions
historical = df[df["InvoiceDate"] < split_date].copy()
evaluation = df[df["InvoiceDate"] >= split_date].copy()

print(f"Historical (Training) Period: {historical['InvoiceDate'].min().date()} to {historical['InvoiceDate'].max().date()}")
print(f"  -> Rows: {len(historical):,}")
print(f"  -> Unique customers: {historical['CustomerID'].nunique():,}")

print()
print(f"Evaluation (Testing) Period: {evaluation['InvoiceDate'].min().date()} to {evaluation['InvoiceDate'].max().date()}")
print(f"  -> Rows: {len(evaluation):,}")
print(f"  -> Unique customers: {evaluation['CustomerID'].nunique():,}")


Historical (Training) Period: 2009-12-01 to 2011-05-31
  -> Rows: 530,472
  -> Unique customers: 4,933

Evaluation (Testing) Period: 2011-06-01 to 2011-12-09
  -> Rows: 248,953
  -> Unique customers: 3,547


### Step 3: Customer Features (RFM Profile)
For every customer in our historical data, we calculate:
- **`total_orders`** (Frequency): How many separate orders did they place?
- **`total_spending`** (Monetary): Total money spent in £.
- **`avg_order_value`**: Average spend per order.
- **`total_quantity`**: Total number of items purchased.
- **`unique_products`**: How many distinct products did they try?
- **`recency_days`** (Recency): How many days before June 1, 2011 was their most recent purchase?
- **`tenure_days`**: How many days have they been an active customer?


In [3]:
# Group by CustomerID and aggregate
customer_features = historical.groupby("CustomerID").agg(
    total_orders=("InvoiceNo", "nunique"),
    total_spending=("TotalAmount", "sum"),
    total_quantity=("Quantity", "sum"),
    unique_products=("StockCode", "nunique"),
    first_purchase=("InvoiceDate", "min"),
    last_purchase=("InvoiceDate", "max"),
    country=("Country", lambda s: s.mode().iloc[0] if len(s) > 0 else "Unknown")
).reset_index()

# Calculate derived metrics
customer_features["avg_order_value"] = (customer_features["total_spending"] / customer_features["total_orders"]).round(2)
customer_features["total_spending"] = customer_features["total_spending"].round(2)

# Days between last purchase and split date (recency)
customer_features["recency_days"] = (split_date - customer_features["last_purchase"]).dt.days
customer_features["tenure_days"] = (split_date - customer_features["first_purchase"]).dt.days

print(f"Generated features for {len(customer_features):,} customers.")
customer_features.head(5)


Generated features for 4,933 customers.


,CustomerID,total_orders,total_spending,total_quantity,unique_products,first_purchase,last_purchase,country,avg_order_value,recency_days,tenure_days
0,12346,12,77556.46,74285,27,2009-12-14 08:34:00,2011-01-18 10:01:00,United Kingdom,6463.04,133,533
1,12347,4,2434.96,1626,90,2010-10-31 14:20:00,2011-04-07 10:43:00,Iceland,608.74,54,212
2,12348,4,1709.40,2497,25,2010-09-27 14:59:00,2011-04-05 10:47:00,Finland,427.35,56,246
3,12349,3,2671.14,993,90,2010-04-29 13:20:00,2010-10-28 08:23:00,Italy,890.38,215,397
4,12350,1,334.40,197,17,2011-02-02 16:01:00,2011-02-02 16:01:00,Norway,334.40,118,118


### Step 4: Product Features (Catalog Performance)
For every product in our historical data, we calculate:
- **`total_quantity_sold`**: Total units sold.
- **`total_revenue`**: Total money brought in.
- **`total_orders`**: How many orders included this product.
- **`unique_customers`**: How many different customers bought it.
- **`avg_unit_price`**: Average selling price.
- **`popularity_score`**: Percentile rank from 0 (least popular) to 100 (most popular bestseller).


In [4]:
product_features = historical.groupby("StockCode").agg(
    description=("Description", lambda s: s.dropna().mode().iloc[0] if not s.dropna().empty else "Unknown Product"),
    total_quantity_sold=("Quantity", "sum"),
    total_revenue=("TotalAmount", "sum"),
    total_orders=("InvoiceNo", "nunique"),
    unique_customers=("CustomerID", "nunique"),
    avg_unit_price=("UnitPrice", "mean")
).reset_index()

product_features["total_revenue"] = product_features["total_revenue"].round(2)
product_features["avg_unit_price"] = product_features["avg_unit_price"].round(2)

# Popularity percentile score (0 to 100)
product_features["popularity_score"] = (product_features["total_orders"].rank(pct=True) * 100).round(1)

print(f"Generated features for {len(product_features):,} products.")
product_features.sort_values(by="total_orders", ascending=False).head(5)


Generated features for 4,235 products.


,StockCode,description,total_quantity_sold,total_revenue,total_orders,unique_customers,avg_unit_price,popularity_score
3690,85123A,WHITE HANGING HEART T-LIGHT HOLDER,75435,201677.23,3916,1313,2.86,100.0
1607,22423,REGENCY CAKESTAND 3 TIER,18478,212307.90,2464,1061,12.42,100.0
3668,85099B,JUMBO BAG RED RETROSPOT,65615,115456.96,2252,785,1.92,100.0
599,21212,PACK OF 72 RETRO SPOT CAKE CASES,74731,35916.88,1978,942,0.54,99.9
3429,84879,ASSORTED COLOUR BIRD ORNAMENT,56446,89778.46,1855,781,1.67,99.9


### Step 5: Customer-Product Interaction Features
Does a specific customer have a history with a specific product?
- **`purchase_count`**: How many times has this customer bought this product?
- **`total_qty_purchased`**: Total units of this product bought by this customer.
- **`has_previous_purchase`**: `1` if bought more than once (repeat buyer), `0` otherwise.
- **`days_since_last_purchase`**: Days since this customer last bought this item.


In [5]:
cp_features = historical.groupby(["CustomerID", "StockCode"]).agg(
    purchase_count=("InvoiceNo", "nunique"),
    total_qty_purchased=("Quantity", "sum"),
    total_spent_on_product=("TotalAmount", "sum"),
    last_purchase_date=("InvoiceDate", "max")
).reset_index()

# Days since this customer last bought this specific product
cp_features["days_since_last_purchase"] = (split_date - cp_features["last_purchase_date"]).dt.days
cp_features["has_previous_purchase"] = (cp_features["purchase_count"] > 1).astype(int)
cp_features["total_spent_on_product"] = cp_features["total_spent_on_product"].round(2)

print(f"Generated {len(cp_features):,} customer-product interaction pairs.")
cp_features.head(5)


Generated 346,010 customer-product interaction pairs.


,CustomerID,StockCode,purchase_count,total_qty_purchased,total_spent_on_product,last_purchase_date,days_since_last_purchase,has_previous_purchase
0,12346,15056BL,1,1,5.95,2010-03-02 13:08:00,455,0
1,12346,15056N,1,1,5.95,2010-03-02 13:08:00,455,0
2,12346,15056P,1,1,5.95,2010-03-02 13:08:00,455,0
3,12346,20679,1,1,5.95,2010-03-02 13:08:00,455,0
4,12346,20682,1,1,3.25,2010-03-02 13:08:00,455,0


### Step 6: Summary & Quality Checks
Let's verify our feature tables:
1. Are there any missing values?
2. Did any future dates sneak into our historical features?


In [6]:
print("--- Sanity Checks ---")
print(f"Customer Features Nulls: {customer_features[['CustomerID', 'total_spending', 'recency_days']].isnull().sum().sum()}")
print(f"Product Features Nulls:  {product_features[['StockCode', 'total_orders']].isnull().sum().sum()}")
print(f"CP Features Nulls:       {cp_features[['CustomerID', 'StockCode', 'purchase_count']].isnull().sum().sum()}")

# Check data leakage: latest date in historical features must be < June 1, 2011
max_hist_date = customer_features["last_purchase"].max()
print()
print(f"Latest date in customer features: {max_hist_date}")
print(f"Split cutoff date:                {split_date}")
if max_hist_date < split_date:
    print("SUCCESS: Zero data leakage! All features are strictly historical.")
else:
    print("WARNING: Data leakage detected!")


--- Sanity Checks ---
Customer Features Nulls: 0
Product Features Nulls:  0
CP Features Nulls:       0

Latest date in customer features: 2011-05-31 15:41:00
Split cutoff date:                2011-06-01 00:00:00
SUCCESS: Zero data leakage! All features are strictly historical.
